# Data Conversion of SBE 911plus .hex data
This python notebook shows how to convert a .hex file into pandas dataframe with scientific values.

Please contact [SBS customer support](https://www.seabird.com/support) for help or to request additional features.

Example data provided by CalCOFI. Station Santa Barbara Basin

## Init

The initialization code section below is used to import required libraries.

In [ ]:
# Native imports
from pathlib import Path

# relative path imports
import example_data.example_coefficients as ec

# Third-party imports
import gsw
import numpy as np
import xarray as xr

# Sea-Bird imports
import seabirdscientific.conversion as sc
import seabirdscientific.instrument_data as si
import seabirdscientific.visualization as sv

## [Data Conversion](#proc-list)

This section shows how to convert raw data contained in a .hex file into scientific units for the instruments that follow:  
- 9/11plus

In [ ]:
hex_file = Path("example_data/SBE911plus/2507054.hex")

# Convert raw hexadecimal string to raw frequencies
raw_data = si.read_hex_file(
    filepath=hex_file,
    instrument_type=si.InstrumentType.SBE911Plus,
    enabled_sensors=[
        si.Sensors.Temperature,
        si.Sensors.Conductivity,
        si.Sensors.Pressure,
        si.Sensors.SecondaryTemperature,
        si.Sensors.SecondaryConductivity,
        si.Sensors.ExtVolt0,
        si.Sensors.ExtVolt1,
        si.Sensors.ExtVolt2,
        si.Sensors.ExtVolt3,
        si.Sensors.ExtVolt4,
        si.Sensors.ExtVolt5,
        si.Sensors.ExtVolt6,
        si.Sensors.ExtVolt7,
        si.Sensors.SPAR,
        si.Sensors.nmeaLocation,
        si.Sensors.SystemTime,
    ],
    frequency_channels_suppressed=0,
    voltage_words_suppressed=0,
)

In [ ]:
# Convert raw frequencies to scientific values
SAMPLE_INTERVAL = 1 / 24

temperature = sc.convert_temperature_frequency(
    frequency=raw_data["temperature"].values,
    coefs=ec.temperature_coefs_sn5102,
    standard="ITS90",
    units="C",
)

pressure_dbar = sc.convert_pressure_digiquartz(
    pressure_count=raw_data["digiquartz pressure"].values,
    compensation_voltage=raw_data["temperature compensation"].values,
    coefs=ec.pressure_coefs_sn0936,
    units="dbar",
    sample_interval=SAMPLE_INTERVAL,
)

conductivity = sc.convert_conductivity(
    conductivity_count=raw_data["conductivity"].values,
    temperature=temperature,
    pressure=pressure_dbar,
    coefs=ec.conductivity_coefs_sn3569,
    scalar=0.1,
)

secondary_temperature = sc.convert_temperature_frequency(
    frequency=raw_data["temperature"].values,
    coefs=ec.temperature_coefs_sn5109,
    standard="ITS90",
    units="C",
)

secondary_conductivity = sc.convert_conductivity(
    conductivity_count=raw_data["conductivity"].values,
    temperature=temperature,
    pressure=pressure_dbar,
    coefs=ec.conductivity_coefs_sn2206,
    scalar=0.1,
)

salinity = gsw.SP_from_C(
    C=conductivity / 1000,  # mS/cm
    t=temperature,  # ITS90 C
    p=pressure_dbar,  # dbar
)

chlorophyll = sc.convert_eco(raw=raw_data["volt 1"].values, coefs=ec.chlorophyll_a_coefs_sn3122)

height = sc.convert_altimeter(volts=raw_data["volt 2"].values, coefs=ec.altimeter_coefs_sn46604)

oxygen = sc.convert_sbe43_oxygen(
    voltage=raw_data["volt 4"].values,
    temperature=temperature,
    pressure=pressure_dbar,
    salinity=salinity,
    coefs=ec.oxygen_43_coefs_sn1590,
    apply_tau_correction=True,
    apply_hysteresis_correction=True,
    window_size=1,
    sample_interval=SAMPLE_INTERVAL,
)

oxygen_secondary = sc.convert_sbe43_oxygen(
    voltage=raw_data["volt 5"].values,
    temperature=temperature,
    pressure=pressure_dbar,
    salinity=salinity,
    coefs=ec.oxygen_43_coefs_sn0680,
    apply_tau_correction=True,
    apply_hysteresis_correction=True,
    window_size=1,
    sample_interval=SAMPLE_INTERVAL,
)

ph = sc.convert_sbe18_ph(
    raw_ph=raw_data["volt 7"].values, temperature=temperature, coefs=ec.ph_coefs_sn0709
)

spar = sc.convert_spar_biospherical(
    volts=raw_data["surface par"].values, coefs=ec.spar_coefs_sn20659
)


# Flag to be used in data processing
flag = np.zeros(len(temperature))

dataset = xr.Dataset(
    coords={"scan": np.arange(len(temperature))},
    data_vars={
        "temperature": ("scan", temperature),
        "conductivity": ("scan", conductivity),
        "pressure": ("scan", pressure_dbar),
        "secondary_temperature": ("scan", secondary_temperature),
        "secondary_conductivity": ("scan", secondary_conductivity),
        "chlorophyll": ("scan", chlorophyll),
        "height": ("scan", height),
        "oxygen": ("scan", oxygen),
        "oxygen_secondary": ("scan", oxygen_secondary),
        "ph": ("scan", ph),
        "spar": ("scan", spar),
        "nmea_lat": ("scan", raw_data["NMEA Latitude"].values),
        "nmea_long": ("scan", raw_data["NMEA Longitude"].values),
        "pump_status": ("scan", raw_data["SBE911 pump status"].values),
        "bottom_contact": ("scan", raw_data["SBE911 bottom contact status"].values),
        "confirm_status": ("scan", raw_data["SBE911 confirm status"].values),
        "modem_status": ("scan", raw_data["SBE911 modem status"].values),
        "system_time": ("scan", raw_data["system time"].values),
        "data_integrity": ("scan", raw_data["data integrity"].values),
        "flag": ("scan", flag),
    },
    attrs={
        "file_name": "2507054.hex",
        "sample_interval": SAMPLE_INTERVAL,
    },
)

dataset

## [Data Plotting](#proc-list)

In [ ]:
config = sv.ChartConfig(
    title="9/11plus Data Conversion",
    x_names=["temperature", "conductivity", "chlorophyll", "oxygen"],
    y_names=["pressure"],
    z_names=[],
    chart_type="overlay",
    plot_loop_edit_flags=False,
    lift_pen_over_bad_data=True,
)

fig = sv.plot_xy_chart(dataset, config)

# plotly customizations
fig["layout"]["yaxis"]["autorange"] = "reversed"
fig.data[0].name = "Temperature"
fig.data[1].name = "Conductivity"
fig.data[2].name = "Chlorophyll"
fig.data[3].name = "Oxygen"
fig["layout"]["yaxis"]["title"] = "Pressure [dbar]"
fig["layout"]["xaxis"]["title"] = "Temperature [ITS-90 degrees C]"
fig["layout"]["xaxis2"]["title"] = "Conductivity [S/m]"
fig["layout"]["xaxis3"]["title"] = "Chlorophyll"
fig["layout"]["xaxis4"]["title"] = "Oxygen [ml/l]"

fig.update_layout(height=800)
fig.show()

## Processing

This converted data should now be processed using the tools in processing.ipynb to produce a final data product.